# CrisisOps v2 — GRPO Training Notebook

Train an LLM to detect deceptive software engineers and recover failing projects.

**One-click in Colab.** Run cells top to bottom.

## What this demonstrates
- **Theme 1**: Two-agent interaction — GRPO-trained PM vs LLM-powered deceptive member
- **Theme 2**: Long-horizon planning with memory buffer compression every 8 steps
- **Theme 3.1**: Real tool use — Jira adapter, observable signal queries, counterfactual reward
- **Theme 4**: Adaptive curriculum — crisis generator tracks agent weaknesses via EMA

**Novel mechanisms implemented:**
1. Dynamic candor evolution — caught liars become more honest; unchecked liars grow bolder
2. Social testimony graph — query one member about another; honest vs allied deceptive give different signals
3. Alibi coordination — deceptive alliance members give consistent coordinated alibis
4. Political capital — second earned resource; spend to compel truth or trigger whistleblower
5. LLM-powered adversarial agent — Ollama (qwen2.5:3b) with OpenAI fallback, or rule-based; contextual adaptive lies
6. Long-horizon memory buffer — episode state compressed every N steps and injected into observation

In [ ]:
# Cell 1: Install dependencies
# Note: unsloth install may take 3-5 minutes on first run
import subprocess, sys

packages = [
    'unsloth',
    'trl>=0.29.0',
    'transformers>=4.46.0',
    'datasets>=3.2.0',
    'accelerate>=1.1.0',
    'peft>=0.13.0',
    'matplotlib',
    'openai>=1.0.0',
    'scipy',
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✓ All dependencies installed')

In [ ]:
# Cell 2: Clone repo and set up environment
import os, sys

IN_COLAB = 'google.colab' in str(get_ipython())

# !! CHANGE THIS to your actual repo URL before running !!
REPO_URL = 'https://github.com/aryannzzz/crisisops'

if IN_COLAB:
    import shutil
    repo_path = '/content/crisisops'
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    os.system(f'git clone {REPO_URL} {repo_path}')
    os.chdir(repo_path)
else:
    # Local: assume already in repo root
    pass

# Add repo root to Python path
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'Working directory: {os.getcwd()}')

# Verify core modules load
from env.environment import CrisisOpsEnv
from env.actions import ACTION_COSTS
print(f'✓ Environment loaded — {len(ACTION_COSTS)} actions registered')
assert len(ACTION_COSTS) >= 16, f'Expected 16+ actions, got {len(ACTION_COSTS)}'

# Verify new mechanisms
from env.state import TeamMember
required_fields = ['alliance_id', 'times_cross_verified', 'caught_this_episode',
                   'is_llm_agent', 'pm_actions_toward_member', 'prior_statements']
tm_fields = set(TeamMember.__dataclass_fields__.keys())
for f in required_fields:
    assert f in tm_fields, f'Missing TeamMember field: {f}'
print('✓ All TeamMember fields present')

from env.state import ProjectState
required_ps = ['political_capital', 'memory_buffer', 'memory_compression_interval']
ps_fields = set(ProjectState.__dataclass_fields__.keys())
for f in required_ps:
    assert f in ps_fields, f'Missing ProjectState field: {f}'
print('✓ All ProjectState fields present')

# Quick smoke test
from scenarios.level1 import get_random_level1_scenario
from reward.counterfactual import counterfactual_reward
env = CrisisOpsEnv(scenario_fn=get_random_level1_scenario(), curriculum_level=1)
obs = env.reset(seed=42)
assert 'political_capital' in obs, 'political_capital missing from observation'
obs2, r, done, info = env.step({'action_type': 'query_status', 'params': {}})
print(f'✓ Smoke test passed — obs keys: {list(obs.keys())}')
print(f'  PC={obs["political_capital"]}, budget={obs["budget_remaining"]}')

In [ ]:
# Cell 3: Calibration — must pass before training
# Expected: greedy 0.45-0.55, oracle 0.70-0.80, gap 0.20-0.35
# Warnings (not failures) are OK.
from calibration.calibrate import run_calibration

print('Running calibration (20 episodes)...')
result = run_calibration(n_episodes=20, seed=99)
print(f"Greedy mean: {result['greedy_mean']:.3f}")
print(f"Oracle mean: {result['oracle_mean']:.3f}")
print(f"Gap: {result['gap']:.3f}")
print(f"Status: {result['status']}")

# Allow warnings; only fail on explicit FAIL status
if 'FAIL' in result.get('status', '').upper() and 'PASS' not in result.get('status', '').upper():
    raise RuntimeError(f'Calibration FAILED: {result}')
print('✓ Calibration OK — proceeding to training')

In [ ]:
# Cell 4: GRPO Training
# 
# For judges: set NUM_EPISODES=300 for a real training run (~2-3h on T4)
# For quick demo: set NUM_EPISODES=50 (~20min on T4)
#
# The trainer uses:
#   - Unsloth 4-bit Qwen2.5-1.5B-Instruct with LoRA (r=16, alpha=32)
#   - TRL GRPOTrainer with counterfactual reward
#   - Adaptive curriculum: levels unlock at reward thresholds 0.15/0.25/0.35
#   - CrisisGenerator samples weakest crisis types most frequently

import os

NUM_EPISODES = 300   # <-- change to 50 for quick demo, 300+ for full training
OUTPUT_DIR = './crisisops_run'
SEED = 42
START_LEVEL = 1

# --- LLM deceptive agent setup (optional, enhances training diversity) ---
# Option A: Ollama local (FREE, recommended)
#   1. Install Ollama: https://ollama.com/download
#   2. Run: ollama pull qwen2.5:3b
#   3. Run: ollama serve  (starts at localhost:11434 automatically)
#   The agent auto-detects Ollama — no env var needed.
#
# Option B: OpenAI API (costs money, ~$0.001 per episode)
#   os.environ['OPENAI_API_KEY'] = 'sk-...'
#
# Option C: No LLM (rule-based fallback, always works, no setup)
#   Just leave both options commented out.
print(f'LLM deceptive agent: checking providers...')
from env.llm_deceptive_agent import _resolve_provider, reset_provider_cache
reset_provider_cache()
provider = _resolve_provider()
print(f'  Selected provider: {provider}')

print(f'Starting GRPO training: {NUM_EPISODES} episodes, level {START_LEVEL}')
print(f'Output: {OUTPUT_DIR}')
print()

from training.grpo_trainer import train
train(
    curriculum_level=START_LEVEL,
    num_episodes=NUM_EPISODES,
    output_dir=OUTPUT_DIR,
    seed=SEED,
)
print('\n✓ Training complete')

In [ ]:
# Cell 5: Training curve visualization
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter1d

OUTPUT_DIR = './crisisops_run'
log_path = os.path.join(OUTPUT_DIR, 'reward_log.json')

if not os.path.exists(log_path):
    raise FileNotFoundError(f'reward_log.json not found at {log_path}. Run Cell 4 first.')

with open(log_path) as f:
    log = json.load(f)

episodes = [r['episode'] for r in log]
rewards  = [r['reward']  for r in log]
levels   = [r['level']   for r in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Reward curve with curriculum shading ---
ax = axes[0]
smoothed = gaussian_filter1d(rewards, sigma=3)
ax.plot(episodes, rewards, alpha=0.25, color='#2563eb', linewidth=0.8, label='Raw CF reward')
ax.plot(episodes, smoothed, color='#1d4ed8', linewidth=2.0, label='Smoothed')
ax.axhline(0, color='#dc2626', linestyle='--', linewidth=1.0, label='Greedy baseline (0)')

# Shade curriculum level regions
level_colors = {1: '#eff6ff', 2: '#f0fdf4', 3: '#fefce8', 4: '#fff1f2'}
level_names  = {1: 'L1', 2: 'L2', 3: 'L3', 4: 'L4'}
prev_ep, prev_lv = (episodes[0] if episodes else 0), (levels[0] if levels else 1)
for ep, lv in zip(episodes, levels):
    if lv != prev_lv:
        ax.axvspan(prev_ep, ep, alpha=0.4, color=level_colors.get(prev_lv, '#f5f5f5'))
        ax.text((prev_ep + ep) / 2, ax.get_ylim()[0] + 0.02,
                level_names[prev_lv], ha='center', fontsize=8, color='#6b7280')
        prev_ep, prev_lv = ep, lv
ax.axvspan(prev_ep, episodes[-1] if episodes else 0,
           alpha=0.4, color=level_colors.get(prev_lv, '#f5f5f5'))

ax.set_xlabel('Training episode', fontsize=11)
ax.set_ylabel('Counterfactual reward (agent − greedy PM)', fontsize=11)
ax.set_title('CrisisOps v2 — GRPO Training Curve', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Reward distribution (trained vs untrained baseline) ---
ax2 = axes[1]
n = len(rewards)
first_quarter = rewards[:n//4]
last_quarter  = rewards[3*n//4:]

import numpy as np
ax2.hist(first_quarter, bins=20, alpha=0.6, color='#dc2626', label=f'First 25% of training (mean={np.mean(first_quarter):.3f})')
ax2.hist(last_quarter,  bins=20, alpha=0.6, color='#2563eb', label=f'Last 25% of training (mean={np.mean(last_quarter):.3f})')
ax2.axvline(0, color='black', linestyle='--', linewidth=1.0, label='Greedy baseline')
ax2.set_xlabel('Counterfactual reward', fontsize=11)
ax2.set_ylabel('Episode count', fontsize=11)
ax2.set_title('Reward Distribution: Early vs Late Training', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve_final.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Episodes trained: {len(episodes)}')
print(f'Mean reward (all): {sum(rewards)/len(rewards):.4f}')
print(f'Mean reward (last 25%): {sum(last_quarter)/len(last_quarter):.4f}')
print(f'Curriculum levels reached: {sorted(set(levels))}')
print(f'Plot saved to: {OUTPUT_DIR}/training_curve_final.png')

In [ ]:
# Cell 6: Qualitative evaluation — trained vs untrained
# Shows the deception detection capability side-by-side

import json, os
from env.environment import CrisisOpsEnv
from scenarios.level3 import scenario_adversarial_majority
from reward.baseline import GreedyPMBaseline

OUTPUT_DIR = './crisisops_run'
EVAL_SEED = 999  # Different seed from training

def run_greedy_episode(seed):
    env = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=3)
    obs = env.reset(seed=seed)
    agent = GreedyPMBaseline()
    done = False
    actions = []
    while not done:
        action = agent.act(env._state)
        obs, reward, done, info = env.step(action)
        actions.append(action.get('action_type', '?'))
    return reward, actions, info

print('=== GREEDY PM BASELINE ===')
g_reward, g_actions, g_info = run_greedy_episode(EVAL_SEED)
print(f'Final counterfactual reward: {g_reward:.4f}')
print(f'Actions taken: {g_actions}')
print(f'Cross-verify rate: {g_info["cross_verify_rate"]:.2f}')
print(f'Active crises at end: {g_info["active_crises"]}')
print()

# Try to load trained model for comparison
checkpoint_path = os.path.join(OUTPUT_DIR, 'checkpoint_ep50')
if os.path.exists(checkpoint_path):
    print('=== TRAINED AGENT (checkpoint_ep50) ===')
    print('(Loading trained model for comparison...)')
    try:
        from unsloth import FastLanguageModel
        from training.grpo_trainer import format_observation_as_prompt, parse_action_from_response, SYSTEM_PROMPT
        import torch

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint_path,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(model)

        env = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=3)
        obs = env.reset(seed=EVAL_SEED)
        done = False
        actions = []
        step = 0

        while not done and step < 30:
            user_content = format_observation_as_prompt(obs)
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_content},
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, temperature=0.1,
                                     do_sample=True, pad_token_id=tokenizer.eos_token_id)
            response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            action = parse_action_from_response(response)
            obs, reward, done, info = env.step(action)
            actions.append(action.get('action_type', '?'))
            step += 1

        print(f'Final counterfactual reward: {reward:.4f}')
        print(f'Actions taken: {actions}')
        print(f'Cross-verify rate: {info["cross_verify_rate"]:.2f}')
        print(f'Active crises at end: {info["active_crises"]}')
        print(f'Improvement over greedy: {reward - g_reward:+.4f}')
    except Exception as e:
        print(f'Could not load trained model: {e}')
        print('Run Cell 4 first to generate a checkpoint.')
else:
    print(f'No checkpoint found at {checkpoint_path}.')
    print('Run Cell 4 to train first, then re-run this cell.')

In [ ]:
# Cell 7: Jira/Linear adapter demo
# Shows real tool-use capability (dry_run=True by default)
from deployment.jira_adapter import JiraAdapter

adapter = JiraAdapter(dry_run=True)

# Simulate what the agent would write to Jira after a successful recovery
recovery_plan = {
    'plan_summary': (
        'CrisisOps agent detected 2 deceptive engineers (alibi coordination identified). '
        'Reassigned 3 critical-path tasks. Used force_truth on dev_am1 (actual=0.08 vs reported=0.72). '
        'Triggered whistleblower tip at PC=8 to identify worst liar. '
        'Client satisfaction maintained at 7.8. All 3 crises resolved.'
    ),
    'risk_items': [
        'dev_am1 and dev_am2 formed alibi alliance — monitoring continues',
        'Schema drift event at step 8 acknowledged within window',
        'Budget consumed 18/20 — future episodes need earlier submission',
    ],
    'timeline': '2026-05-15',
}

adapter.submit_recovery_plan(recovery_plan)
print('✓ Recovery plan submitted to Jira (dry_run=True — set dry_run=False with real credentials to write live)')